(html-intro_structural-analysis)=
# 🚀 Quellcode-Analyse einer Website 

## Hinweise zur Ausführung des Notebooks
Dieses Notebook kann auf unterschiedlichen Levels erarbeitet werden (siehe Abschnitt ["Technische Voraussetzungen"](../introduction/introduction_requirements)): 
1. Book-Only Mode
2. Cloud Mode: Dafür auf 🚀 klicken und z.B. in Colab ausführen.
3. Local Mode: Dafür auf Herunterladen ↓ klicken und ".ipynb" wählen. 

## Übersicht
Im Folgenden wird exemplarisch der HTML-Code der Website der Senatskanzlei Berlin auf seine Struktur hin untersucht und es wird eine strukturierte Methode zur Inhaltsextraktion entwickelt.

Dafür werden folgende Schritte durchgeführt:
1. Strukturanalyse des HTML-Codes
2. Strukturiertes Parsen des HTML-Codes
3. Verlinkten Seiten nachgehen und parsen
4. Ergebnisse speichern

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
  
<b>Voraussetzungen zur Ausführung des Jupyter Notebooks:</b>
<ul>
<li> Installieren der Bibliotheken </li>
</ul>
Zum Testen: Ausführen der Zelle "load libraries".</br>
Alle Zellen, die mit 🚀 gekennzeichnet sind, werden nur bei der Ausführung des Notebooks in Colab / JupyterHub bzw. lokal ausgeführt. 
</details>

In [ ]:
#  🚀 Install libraries 
! pip install requests beautifulsoup4 pandas

In [ ]:
# load libraries
from datetime import datetime
from pathlib import Path

import requests
from bs4 import BeautifulSoup, Tag, Comment
import pandas as pd

## Laden des HTML-Codes 

Im Folgenden laden wir den HTML Code der Website des Berliner Senats ([https://www.berlin.de/rbmskzl/](https://www.berlin.de/rbmskzl/)) vom 06.06.2025, den wir im Vorhinein in einer `.html`-Datei gespeichert haben. 

<details>
  <summary><b>Informationen zum Ausführen des Notebooks – Zum Ausklappen klicken ⬇️</b></summary>
Zuerst wird der Ordner angelegt, in dem die HTML-Datei gespeichert wird. Der Einfachheit halber wird die gleiche Datenablagestruktur wie in dem <a href="https://github.com/quadriga-dk/Text-Fallstudie-2/tree/main">GitHub Repository</a>, in dem die Daten gespeichert sind, vorausgesetzt. </br>
Der Text wird aus GitHub heruntergeladen und in dem Ordner <i>../data/html/</i> abgespeichert. </br>
Der Pfad kann in der Variable <i>text_path</i> angepasst werden. Die einzulesenden Daten müssen die Endung `.html` haben. </br>
</details>

In [ ]:
# 🚀 Create data directory path
corpus_dir = Path("../data/html")
corpus_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 🚀 Load the html file from GitHub 
! wget https://raw.githubusercontent.com/quadriga-dk/Text-Fallstudie-2/refs/heads/main/data/html/2025-06-06-Senatskanzlei.html -P ../data/txt

In [ ]:
# Set file paths 
path_to_html_doc = Path("../data/html/2025-06-06-Senatskanzlei.html")
# Read the text
html_text = path_to_html_doc.read_text()
# Parse the html structure
soup = BeautifulSoup(html_text)

Der obere Teil der Website und der korrespondierende HTML-Code sahen zum Zeitpunkt der Speicherung so aus:

![Website des Berliner Senats, 06.06.2025](../assets/images/Website-Senat-Aktuelles.png)
![Entsprechender HTML-Ausschnitt, 06.06.2025](../assets/images/HTML-Dokument.png) 

Orange Markierungen zeigen in welchen HTML-Tags der sichtbare Text gespeichert ist. Blaue Markierungen zeigen an, worauf die Links unter "Weitere Informationen" verweisen.

## Strukturelle Analyse
### Vorgehen
Im nächsten Schritt soll ein kleines Programm entwickelt werden, dass den Text der Website sowie die Links zu den vollen Artikeln extrahiert. Da der Text schon in einer strukturierten Form vorliegt, soll von dieser Gebrauch gemacht werden und Titel von Teaser getrennt extrahiert werden. 

Wir können mit Hilfe der Python-Bibliothek `beautifulsoup` die geschachtelte Struktur des HTML-Codes navigieren. Dafür gucken wir zuerst:
1. Ist die visuelle Aufteilung der Seite in den Tags abgebildet?
2. Welche Tags (mit Attribut) unterteilen die Abschnitte?
3. Sind die Tags für den gegebenen Abschnitt einzigartig?
4. Wie sind die Tags hierarchisch strukturiert?

### Ausschnitt identifizieren
Wir sehen, dass der links abgebildete Inhalt dem div-Container `<div>`  mit CSS class `'herounit-homepage herounit-homepage--default'` untergeordnet ist und können diesen und alle untergeordneten Tags (sogenannte "children") mit `beautifulsoup` extrahieren.


In [ ]:
# get all tags that are children of the div tag with matching CSS class
topdiv = soup.find("div", {"class": "herounit-homepage herounit-homepage--default"})

# print the content of the topdiv
print(topdiv.prettify())

### Titel extrahieren

Wir sehen, dass alle Überschriften unter `h2`-Tags stehen. Diese können wir im nächsten Schritt extrahieren. Wir gehen dabei von dem bereits extrahierten Top-Div aus und extrahieren nur `h2`-Tags, die diesem Tag untergeordnet sind. 

In [ ]:
topdiv_h2titles = topdiv.find_all('h2')

In [ ]:
# get all h2 content that is a child of the top div
topdiv_h2titles = topdiv.find_all('h2')

# retrieve the content and clean it
topdiv_h2titles = [entry.text.strip() for entry in topdiv_h2titles]

print(topdiv_h2titles)

### Kurzbeschreibungen extrahieren 

Wir sehen weiter, dass alle Kurzbeschreibungen als paragraphs `<p>` ausgezeichnet sind. Im Folgenden extrahieren wir alle Paragraphen und lassen uns das Ergebnis anzeigen.


In [ ]:
# get all paragraphs that are children of the top div
topdiv_texts =  topdiv.find_all('p')
topdiv_texts

Wir extrahieren zwar so alle Kurzbeschreibungen, unsere Liste beinhaltet allerdings auch die Beschreibung eines Bilds. Da sich das `class`-Attribut der Kurzbeschreibung von dem des Bilds unterscheidet, können wir durch das zusätzliche Abgleichen des Attributs eine Liste erstellen, in der nur die Kurzbeschreibungen vorhanden sind: 

In [ ]:
# get all short description from the p for which the attribute "class" equals "text"
topdiv_texts =  topdiv.find_all('p', {"class":"text"})

# retrieve the content and clean it
topdiv_texts = [entry.text.strip() for entry in topdiv_texts]

# print the extracted content
topdiv_texts

### Links extrahieren

Auf die gleiche Weise können wir alle Hyperlinks, die in `<a>`-Tags gespeichert sind extrahieren. Der Hyperlink selbst steht in dem Attribut `href`, dessen Wert wir gezielt abfragen.

In [ ]:
topdiv_links =  topdiv.find_all('a')
topdiv_links = [entry.get('href') for entry in topdiv_links]

# print the extracted links
topdiv_links

Wir sehen, dass die Links keine vollständigen URLs sind, da sie weder mit `www.` noch mit `https://` anfangen. Diese Links nennen wir **relative URLs**. Sie verweisen auf Unterseiten der aktuellen Seite (die Startseite der Senatskanzlei). Die Adresse der Unterseiten wird relativ zur aktuellen Seite angegeben.
Die Abfrage dieser relativen URLs in einem Browser funktioniert nicht, es wird ein `File not found`-Error zurückgegeben, da der Browser versucht eine Datei im lokalen Dateisystem zu öffnen und die angegebene Datei nicht findet. Um die Website abfragen zu können, müssen wir die relativen URLs in absolute URLs umwandeln. Beim Umwandeln wird das Präfix der aktuellen Seite vorangestellt werden, in unserem Fall "https://www.berlin.de/".

In [ ]:
# create absolute URLs
def make_links_absolute(link_list, prefix="https://www.berlin.de"):
    absolute_links = []
    for link in link_list:
        if not link.startswith("https"):
            absolute_links.append(prefix + link)
        else:
            absolute_links.append(link)    
    return absolute_links

In [ ]:
topdiv_absolute_links = make_links_absolute(topdiv_links)
topdiv_absolute_links

## Zusammenfügen der Daten 
Wir haben nun unterschiedliche drei Listen mit zusammenhängende Daten aus dem HTML-Code extrahiert:
* Titel
* Teaser-Text
* URLs zu den vollständigen Artikeln

Diese wollen in einem nächsten Schritt zusammenfügen. Dafür prüfen wir zuerst die Vollständigkeit der Daten, das heißt, ob alle Listen die gleiche Länge haben. Da wir die Daten immer in derselben Reihenfolge abgeschritten sind, können wir die Reihenfolge der Listen nutzen, um sie zusammenzufügen. \
Wir speichern zusätzlich ein weiteres Metadatum und zwar das Datum der Extraktion. \
Die Daten bilden wir in einer Tabelle ab, da sich diese Datenstruktur gut für relationale Daten eignet. 

In [ ]:
# check if lists have the same length
if len(topdiv_h2titles) == len(topdiv_texts) == len(topdiv_absolute_links):
    # create 
    top_section_data = {"DC.title": topdiv_h2titles,
                      "Text":topdiv_texts, 
                      "DC.source":topdiv_absolute_links,
                       "DC.date": datetime.today().strftime('%d.%m.%Y') }
else:
    print("Die Listen haben nicht dieselbe Länge:")
    print(f"Titel: {len(topdiv_h2titles)}; Teaser: {len(topdiv_texts)}; URLS: {len(topdiv_absolute_links)}")
    top_section_data = None

top_section_data_df = pd.DataFrame(top_section_data)
top_section_data_df

Aufbauend auf dieser Tabelle könnten wir nun automatisch die Volltexte der Artikel extrahieren, indem wir die gespeicherten URLs automatisch abfragen und den Text extrahieren. Dafür müssen wir auf Methoden des Web-Scraping (siehe folgendes Kapitel [„Scraping als Methode zum Korpusaufbau“](scraping-intro_intro)) zurückgreifen.

## Ergebnisse speichern 
Schlussendlich speichern wir die Ergebnisse. Da wir keine Volltexte extrahiert haben und somit nur Metadaten extrahiert haben, speichern wir diese gesammelt in einer Tabelle. 

### Ergebnis-Ordner und Dateipfad festlegen

Ordner zum Schreiben der Textdateien festlegen:

In [ ]:
output_dir = Path(r"../data/txt/senatskanzlei")

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

Dateinamen erstellen:

In [ ]:
date = datetime.today().strftime('%Y-%m-%d')
fn = f"{date}_Senatskanzlei_Aktuelles.csv"
fp = output_dir / fn

### Speichern der Daten

In [ ]:
top_section_data_df.to_csv(fp, index=False)